# Decompose — fit on ds002837 + cneuromod, project camcan

Replicates `1. decomposition_dataframe_2.ipynb` on the new pipeline, with four
changes:

1. **Fit and projection are separate.** The old version fit on ds002837 only.
   Here the scaler, PCA, UMAP and the bin edges are fit on the training cohorts
   and *applied* to camcan, so camcan is genuinely held out — and it is the
   cohort the symptom question is about.
2. **The cluster label is fixed.** The old `group_{n}` concatenated bin indices
   as strings, so at `n_bins >= 10` two different cells could collide
   (`(1,12,2)` and `(11,2,2)` both give `"1122"`). `np.ravel_multi_index` gives
   the true cell index instead. Only `n_bins` of 2, 3, 5 and 8 were correct
   before.
3. **Edges are not copied into the output.** The old parquet carried every edge
   alongside the latents (7,422 columns for the 122-node atlas). They already
   live in `dfc/`; this writes identity + latents + clusters only.
4. **KernelPCA is dropped.** Its default kernel is linear, so it was PCA with a
   dense n x n kernel matrix — 11 GB at the old row count, ~135 GB at this one.

Column names are kept exactly as before (`pca0/3`, `umap0/3`,
`ThresholdCluster_pca3_512`) so the old plotting code reads unchanged.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import KBinsDiscretizer, StandardScaler

ROOT = Path(os.environ.get("FMRIDECOMP_OUTPUTS",
                           "/project/6008063/tamires/DecomposingfMRI/outputs"))
ATLAS = "harvardoxford"
WINDOW_S = 30
TRAIN = ["ds002837", "cneuromod"]      # fit here
PROJECT = ["camcan"]                   # transform only -- held out
N_LATENTS = [2, 3, 5]
BINS = [2, 3, 5, 8]                    # threshold grid per PCA axis -> n**3 cells
UMAP_FIT_ROWS = 30_000                 # UMAP is fit on a subsample; 0 = all
SEED = 42

QC = ["window_id", "start_tr", "start_s", "stimulus_start_s", "stimulus_end_s",
      "n_tr_nominal", "n_tr_available", "n_tr_effective", "frac_good_frames",
      "crosses_run_boundary", "crosses_clip_boundary", "rank_deficient",
      "ses", "run", "acq", "run_key"]
IDENT = ["cohort", "task", "sub", "window_id", "start_s", "stimulus_start_s",
         "n_tr_effective", "frac_good_frames", "rank_deficient",
         "crosses_run_boundary"]

OUT = ROOT / "latents" / f"atlas={ATLAS}" / f"window_s={WINDOW_S}"
MODELS = ROOT / "meta" / "models"
print("output_root:", ROOT, "\nlatents ->", OUT)

In [ ]:
def load_cohort(cohort):
    """All dfc windows for one cohort at this atlas and window size."""
    paths = sorted(ROOT.glob(f"dfc/atlas={ATLAS}/window_s={WINDOW_S}/"
                             f"cohort={cohort}/task=*/sub=*/data.parquet"))
    if not paths:
        raise FileNotFoundError(f"no shards for cohort={cohort} at "
                                f"{ATLAS}/{WINDOW_S}s")
    df = pd.concat([pd.read_parquet(p).assign(
                        **dict(s.split("=", 1) for s in p.parts if "=" in s))
                    for p in paths], ignore_index=True)
    return df

frames = {c: load_cohort(c) for c in TRAIN + PROJECT}
EDGES = [c for c in frames[TRAIN[0]].columns if "__" in c]
if not EDGES:
    raise ValueError("no NodeA__NodeB columns -- this atlas packs its edges "
                     "into a single `edges` list column; unpack it first")

for c, d in frames.items():
    print(f"{c:<12} {len(d):>8,} windows  {d['sub'].nunique():>4} subs  "
          f"{d[EDGES].isna().any(axis=1).sum():>6,} rows with a NaN edge  "
          f"rank_deficient {d['rank_deficient'].mean():.1%}")
print(f"\n{len(EDGES)} edges")

In [ ]:
# Drop only what the maths cannot take: a window with any NaN edge. Windows
# flagged rank_deficient are KEPT -- the matrix is singular, but each edge is
# still an ordinary correlation, and dropping them would silently remove whole
# window sizes for the coarse-TR cohort.
clean = {c: d[~d[EDGES].isna().any(axis=1)].reset_index(drop=True)
         for c, d in frames.items()}
for c, d in clean.items():
    print(f"{c:<12} {len(frames[c]):>8,} -> {len(d):>8,} windows  "
          f"({d['sub'].nunique()} subs)")

X_train = pd.concat([clean[c] for c in TRAIN], ignore_index=True)
print(f"\ntraining matrix: {len(X_train):,} x {len(EDGES)} "
      f"= {len(X_train) * len(EDGES) * 8 / 1e9:.1f} GB as float64")

## Fit — scaler, PCA, UMAP, bin edges (training cohorts only)

In [ ]:
scaler = StandardScaler().fit(X_train[EDGES].to_numpy(dtype=np.float64))
Z_train = scaler.transform(X_train[EDGES].to_numpy(dtype=np.float64))

models = {"scaler": scaler, "pca": {}, "umap": {}, "bins": {},
          "edges": EDGES, "atlas": ATLAS, "window_s": WINDOW_S,
          "train_cohorts": TRAIN, "n_train_rows": len(X_train)}

for n in N_LATENTS:
    models["pca"][n] = PCA(n_components=n, random_state=SEED,
                           svd_solver="randomized").fit(Z_train)
    evr = models["pca"][n].explained_variance_ratio_
    print(f"pca {n}: explained {evr.round(3)}  cumulative {evr.sum():.3f}")

In [ ]:
# UMAP is fit on a subsample -- the neighbour graph is O(n log n) but with a
# large constant, and nothing downstream needs it fit on every row.
try:
    import umap
    rng = np.random.default_rng(SEED)
    idx = (rng.choice(len(Z_train), UMAP_FIT_ROWS, replace=False)
           if UMAP_FIT_ROWS and len(Z_train) > UMAP_FIT_ROWS
           else np.arange(len(Z_train)))
    print(f"fitting UMAP on {len(idx):,} of {len(Z_train):,} rows")
    for n in N_LATENTS:
        models["umap"][n] = umap.UMAP(n_components=n, random_state=SEED).fit(Z_train[idx])
        print(f"  umap {n}: done")
except ImportError:
    print("umap-learn not installed -- skipping UMAP, PCA still runs")

In [ ]:
# Threshold clusters, on the 3-component PCA space, fit on the training rows.
# strategy='quantile' as before; ravel_multi_index instead of string
# concatenation, so the label is the real cell index and cannot collide.
if 3 not in N_LATENTS:
    raise ValueError("the threshold grid is defined on pca3; add 3 to N_LATENTS")
P3_train = models["pca"][3].transform(Z_train)

for n in BINS:
    kbd = KBinsDiscretizer(n_bins=n, encode="ordinal", strategy="quantile",
                           subsample=None).fit(P3_train)
    models["bins"][n] = kbd
    lab = np.ravel_multi_index(kbd.transform(P3_train).astype(int).T, (n, n, n))
    print(f"{n} bins -> {n**3:>6} cells, {len(np.unique(lab)):>6} occupied "
          f"({len(np.unique(lab)) / n**3:.0%})")

## Transform every cohort and write

In [ ]:
def latents_for(df):
    """Identity + every latent + every cluster label, for one cohort."""
    Z = scaler.transform(df[EDGES].to_numpy(dtype=np.float64))
    out = df[[c for c in IDENT if c in df.columns]].copy()

    for kind in ("pca", "umap"):
        for n, model in models[kind].items():
            arr = model.transform(Z)
            for j in range(n):
                out[f"{kind}{j}/{n}"] = arr[:, j]

    P3 = models["pca"][3].transform(Z)
    for n, kbd in models["bins"].items():
        out[f"ThresholdCluster_pca3_{n**3}"] = np.ravel_multi_index(
            kbd.transform(P3).astype(int).T, (n, n, n))
    return out

OUT.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

for cohort, df in clean.items():
    lat = latents_for(df)
    path = OUT / f"cohort={cohort}" / "data.parquet"
    path.parent.mkdir(parents=True, exist_ok=True)
    lat.to_parquet(path, index=False)
    role = "train" if cohort in TRAIN else "projected"
    print(f"{cohort:<12} {role:<10} {len(lat):>8,} rows x {lat.shape[1]} cols "
          f"-> {path.relative_to(ROOT)}")

# joblib if the image has it, pickle otherwise -- the file is only ever read
# back by 06, so the format matters less than not failing here.
try:
    import joblib
    model_path = MODELS / f"decompose_atlas-{ATLAS}_window-{WINDOW_S}.joblib"
    joblib.dump(models, model_path)
except ImportError:
    import pickle
    model_path = MODELS / f"decompose_atlas-{ATLAS}_window-{WINDOW_S}.pkl"
    model_path.write_bytes(pickle.dumps(models))
print(f"\nmodels -> {model_path.relative_to(ROOT)}")

In [ ]:
# Sanity: does the held-out cohort land in the same place as the training one?
# A large shift in the PCA means is the acquisition difference (TR 2.47 against
# 1.0 / 1.49), not a brain-state difference -- expected, and the reason the
# symptom analysis stays WITHIN camcan.
check = pd.concat([latents_for(d).assign(cohort=c) for c, d in clean.items()],
                  ignore_index=True)
check.groupby("cohort")[[f"pca{j}/3" for j in range(3)]].agg(["mean", "std"]).round(2)